# 📦 Bronze to Silver ELT Notebook with Schema Drift (Fabric Lakehouse)

This notebook implements the Bronze → Silver step in medallion architecture on Microsoft Fabric.

It is designed to **accumulate full data history with SCD Type 2 columns** while **allowing schema drift** in the Silver layer.

---

## ⚙️ Pipeline Strategy
- **Bronze Layer:** Raw, evolving schema from source systems.
- **Silver Layer:** Accumulated history with SCD2 fields:
  - ELT_SCD_HASH
  - ELT_VALID_FROM
  - ELT_VALID_TO
  - Current_Record_Flag
- Allows **schema drift** by overwriting Silver with new columns automatically.

---

## 📈 Steps
1️⃣ Load Bronze  
2️⃣ Identify new or changed records  
3️⃣ Add ELT_SCD_HASH and SCD2 fields  
4️⃣ Union with existing Silver (if any) to maintain history  
5️⃣ Overwrite Silver table with updated data (and schema)  
6️⃣ Log notebook execution and exit with results

---

## 🎯 Notes
- Fabric Lakehouse tables are compatible with `.option("overwriteSchema", "true")`.
- No MERGE operation required.
- Ideal for pipelines that evolve over time.


## Set Parameters for notebook

In [1]:
# Parameters
p_merge_key_cols = ''
p_src_modified_field = 'ELT_Load_Date_CST'
p_src_db_name = 'Bronze_Lake'
p_src_entity_name = ''
p_tgt_db_name = 'Silver_Lake'
p_tgt_entity_name = ''

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 3, Finished, Available, Finished)

## 📌 Import Necessary Libraries

In [2]:
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import sha2, concat_ws, col, lit, to_date

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 4, Finished, Available, Finished)

## 📌 Config Building and Validation

In [3]:
config = {
    "src_db_name": p_src_db_name,
    "src_entity_name": p_src_entity_name,
    "tgt_db_name": p_tgt_db_name,
    "tgt_entity_name": p_tgt_entity_name,
    "merge_key_cols": p_merge_key_cols,
    "src_modified_field": p_src_modified_field,
    "scd2_valid_from_col": "ELT_VALID_FROM",
    "scd2_valid_to_col": "ELT_VALID_TO",
    "scd2_current_flag_col": "Current_Record_Flag",
    "hash_key_col": "ELT_SCD_HASH",
    "hash_func": "sha2",
    "hash_delim": "||"
}

# Validation
for key in config:
    if config[key] is None:
        raise ValueError(f"Missing required parameter: {key}")

print("✅ Config parameters loaded successfully.")

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 5, Finished, Available, Finished)

✅ Config parameters loaded successfully.


## 📌 Logging Utility

In [4]:
def log(msg):
    """Simple logging function with consistent prefix."""
    print(f"[BronzeToSilver] {msg}")

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 6, Finished, Available, Finished)

## 📌 Load Bronze Data

In [5]:
def load_bronze_data(config):
    table = f"{config['src_db_name']}.{config['src_entity_name']}"
    log(f"Loading Bronze table: {table}")
    df = spark.table(table)
    log(f"Loaded {df.count()} records from Bronze.")
    return df

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 7, Finished, Available, Finished)

## ✅ Get Existing Silver Data (if any)

In [6]:
def load_silver_data(config):
    table = f"{config['tgt_db_name']}.{config['tgt_entity_name']}"
    try:
        df = spark.table(table)
        log(f"Loaded existing Silver table with {df.count()} records.")
        return df
    except Exception:
        log("Silver table does not exist yet. First-time load.")
        return None

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 8, Finished, Available, Finished)

## 📌 Get Max Modified Date from Silver

In [7]:
def get_max_modified_from_silver(silver_df, config):
    if silver_df is None:
        return None

    modified_field = config['src_modified_field']
    df_max = silver_df.select(F.max(F.to_date(modified_field, 'yyyyMMdd')).alias('max_date'))
    max_date = df_max.collect()[0]['max_date']

    if max_date:
        log(f"Max modified date in Silver: {max_date}")
        return datetime.combine(max_date, datetime.min.time())
    else:
        log("Silver table has no max modified date.")
        return None

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 9, Finished, Available, Finished)

## 📌 Filter for New or Changed Records

In [8]:
def filter_new_records(bronze_df, max_modified_date, config):
    modified_field = config['src_modified_field']
    if max_modified_date:
        log(f"Filtering Bronze for {modified_field} > {max_modified_date}")
        filtered = bronze_df.filter(
            F.to_date(F.col(modified_field), 'yyyyMMdd') > F.lit(max_modified_date)
        )
    else:
        log("No existing Silver max date. Using all Bronze records.")
        filtered = bronze_df
    count = filtered.count()
    log(f"Filtered {count} records from Bronze.")
    return filtered

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 10, Finished, Available, Finished)

## 📌 Add HASH_KEY Column

In [9]:
def add_hash_key(df, config):
    """
    Adds HASH_KEY column based on all business columns except excluded ones.
    """
    # Split merge key columns into a set
    merge_keys = set(col.strip() for col in config['merge_key_cols'].split(','))

    # Define columns to exclude from hashing
    exclude_cols = merge_keys.union({
        config['src_modified_field'],
        config['hash_key_col'],
        config['scd2_valid_from_col'],
        config['scd2_valid_to_col'],
        config['scd2_current_flag_col']
    })

    # Determine columns to include in HASH
    hash_cols = [c for c in df.columns if c not in exclude_cols]

    log(f"Generating HASH_KEY from columns: {hash_cols}")

    # Build the HASH_KEY column
    return df.withColumn(
        config['hash_key_col'],
        F.sha2(F.concat_ws(config['hash_delim'], *[F.col(c) for c in hash_cols]), 256)
    )


StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 11, Finished, Available, Finished)

## 📌 Add SCD2 Fields

In [10]:
def add_scd2_fields(df, config):
    valid_from = config['scd2_valid_from_col']
    valid_to = config['scd2_valid_to_col']
    current_flag = config['scd2_current_flag_col']
    modified_field = config['src_modified_field']

    log("Adding SCD2 fields.")
    return (
        df
        .withColumn(valid_from, F.to_date(F.col(modified_field), 'yyyyMMdd'))
        .withColumn(valid_to, F.lit('9999-12-31').cast('date'))
        .withColumn(current_flag, F.lit(1))
    )

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 12, Finished, Available, Finished)

## 📌 Combine Silver and New Bronze with SCD2 Logic

In [11]:
def build_updated_silver(silver_df, bronze_df, config):
    if silver_df is None or silver_df.count() == 0:
        log("No existing Silver records. Using Bronze as initial Silver.")
        return bronze_df

    merge_keys = config['merge_key_cols'].split(',')
    hash_col = config['hash_key_col']
    modified_field = config['src_modified_field']
    valid_to_col = config['scd2_valid_to_col']
    current_flag_col = config['scd2_current_flag_col']

    # Find matched records with changed HASH
    join_cond = [silver_df[k] == bronze_df[k] for k in merge_keys]
    joined = silver_df.filter(F.col(current_flag_col) == 1) \
        .join(bronze_df, join_cond, "inner")

    changed = joined.filter(silver_df[hash_col] != bronze_df[hash_col]) \
        .select([silver_df[c] for c in silver_df.columns]) \
        .withColumn(valid_to_col, F.to_date(F.col(modified_field), 'yyyyMMdd')) \
        .withColumn(valid_to_col, F.date_add(F.col(valid_to_col), 1)) \
        .withColumn(current_flag_col, F.lit(0))

    # Get unchanged Silver records
    unchanged = silver_df.filter(F.col(current_flag_col) == 1) \
        .join(bronze_df, join_cond, "left_anti")

    # All inactive history
    history = silver_df.filter(F.col(current_flag_col) == 0)

    # Combine
    final_silver = (
        history
        .unionByName(unchanged, allowMissingColumns=True)
        .unionByName(changed, allowMissingColumns=True)
        .unionByName(bronze_df, allowMissingColumns=True)
    )
    log(f"Final Silver records after SCD2 logic: {final_silver.count()}")
    return final_silver

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 13, Finished, Available, Finished)

## 📌 Write Silver Table with OverwriteSchema

In [12]:
def write_silver_table(df, config):
    target_table = f"{config['tgt_db_name']}.{config['tgt_entity_name']}"
    log(f"Overwriting Silver table: {target_table} with schema evolution.")
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
    log("✅ Silver table written successfully.")

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 14, Finished, Available, Finished)

## 📌 Orchestration Function

In [13]:
from datetime import datetime

def run_bronze_to_silver_pipeline(config):
    """
    Ingests data from Bronze to Silver with ELT metadata.
    Expects config with keys
    """
    log("🚀 Starting Bronze to Silver pipeline with overwrite strategy.")
    start_time = datetime.now()
    start_iso = start_time.isoformat()
    payload = {}

    try:
        # 1) Load Bronze & Silver
        bronze_df = load_bronze_data(config)
        silver_df = load_silver_data(config)

        # 2) Determine watermark & filter
        max_modified_date = get_max_modified_from_silver(silver_df, config)
        new_bronze_df = filter_new_records(bronze_df, max_modified_date, config)

        # 3) No-new-records branch
        if new_bronze_df.count() == 0:
            log("No new or changed Bronze records to process.")
            end_iso = datetime.now().isoformat()
            payload = {
                "status": "Success",
                "recordsProcessed": 0,
                "startTime": start_iso,
                "endTime": end_iso,
                "errorMessage": "",
                "elt_load_time": max_modified_date.isoformat() if max_modified_date else start_iso,
                "tableName": config["src_entity_name"]
            }
        else:
            # 4) Transform & write
            new_bronze_df = new_bronze_df.cache()
            new_bronze_df = add_hash_key(new_bronze_df, config)
            new_bronze_df = add_scd2_fields(new_bronze_df, config)

            updated_silver_df = build_updated_silver(silver_df, new_bronze_df, config)
            write_silver_table(updated_silver_df, config)

            # 5) Success payload
            record_count = updated_silver_df.count()
            new_max_date = get_max_modified_from_silver(updated_silver_df, config)
            end_iso = datetime.now().isoformat()

            log(f"🏁 Pipeline complete. Returning elt_load_time: {new_max_date}")
            payload = {
                "status": "Success",
                "recordsProcessed": record_count,
                "startTime": start_iso,
                "endTime": end_iso,
                "errorMessage": "",
                "elt_load_time": new_max_date.isoformat() if new_max_date else end_iso,
                "tableName": config["src_entity_name"]
            }

    except Exception as e:
        # 6) Failure payload
        end_iso = datetime.now().isoformat()
        payload = {
            "status": "Failure",
            "recordsProcessed": 0,
            "startTime": start_iso,
            "endTime": end_iso,
            "errorMessage": str(e),
            "elt_load_time": end_iso,
            "tableName": config["src_entity_name"]
        }

    # 7) Exit once with the assembled payload
    mssparkutils.notebook.exit(payload)

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 15, Finished, Available, Finished)

## 📌 Execution Cell

In [14]:
# ✅ Run the full Bronze to Silver process
run_bronze_to_silver_pipeline(config)

StatementMeta(, 962416cf-6357-4450-bda9-09fb90c140bc, 16, Finished, Available, Finished)

[BronzeToSilver] 🚀 Starting Bronze to Silver pipeline with overwrite strategy.
[BronzeToSilver] Loading Bronze table: Bronze_Lake.as_RouteOptimization
[BronzeToSilver] Loaded 88034 records from Bronze.
[BronzeToSilver] Loaded existing Silver table with 175718 records.
[BronzeToSilver] Max modified date in Silver: 2025-07-18
[BronzeToSilver] Filtering Bronze for ELT_Load_Date_CST > 2025-07-18 00:00:00
[BronzeToSilver] Filtered 0 records from Bronze.
[BronzeToSilver] No new or changed Bronze records to process.
ExitValue: {'status': 'Success', 'recordsProcessed': 0, 'startTime': '2025-07-19T02:05:21.013634', 'endTime': '2025-07-19T02:05:47.246501', 'errorMessage': '', 'elt_load_time': '2025-07-18T00:00:00', 'tableName': 'as_RouteOptimization'}

---
✅ Notebook Complete
- Silver table updated with evolving schema and full history.
- elt_load_time returned for downstream pipeline use.